In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# EDIT THESE
# ============================================================
CONTEXT_DIR = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2a_steps1_3_context_review_v4/"
    "contexts/context_011__NAC2020__AR__complete_response__TURBT__all__median"
)

RULES = {
    "min_oof_metric": 0.75,
    "min_delta_clinical": None,
    "max_fold_sd": 0.02,
    "min_direction_consistency": 0.80,
    "min_nonmissing_fraction": 0.50,
    "min_candidate_evidence_score": None,
    "min_valid_folds": 4,
    "max_nominal_p": None,
    "max_context_q": None,
    "max_candidates": 50,
}

# ============================================================
# LOAD
# ============================================================
df = pd.read_parquet(CONTEXT_DIR / "best_transform_features.parquet").copy()

checks = {
    "oof_metric": ("min", RULES["min_oof_metric"]),
    "delta_clinical": ("min", RULES["min_delta_clinical"]),
    "fold_sd": ("max", RULES["max_fold_sd"]),
    "direction_consistency": ("min", RULES["min_direction_consistency"]),
    "nonmissing_fraction": ("min", RULES["min_nonmissing_fraction"]),
    "candidate_evidence_score": ("min", RULES["min_candidate_evidence_score"]),
    "valid_folds": ("min", RULES["min_valid_folds"]),
    "p_value": ("max", RULES["max_nominal_p"]),
    "context_q_value": ("max", RULES["max_context_q"]),
}

# Apply the same AND logic used by Stage 2A-4
pass_cols = []

for column, (direction, threshold) in checks.items():
    pass_col = f"pass_{column}"
    pass_cols.append(pass_col)

    if threshold is None:
        df[pass_col] = True
        continue

    values = pd.to_numeric(df[column], errors="coerce")

    if direction == "min":
        df[pass_col] = values.notna() & (values >= threshold)
    else:
        df[pass_col] = values.notna() & (values <= threshold)

df["passes_all_thresholds"] = df[pass_cols].all(axis=1)

# Stage 2A-4 sorting order
passing = (
    df[df["passes_all_thresholds"]]
    .sort_values(
        ["candidate_evidence_score", "oof_metric",
         "fold_sd", "nonmissing_fraction"],
        ascending=[False, False, True, False],
        na_position="last",
    )
)

selected = passing.head(RULES["max_candidates"]).copy()
selected["seed_rank"] = np.arange(1, len(selected) + 1)

# ============================================================
# SUMMARIES
# ============================================================
print(f"Total best-transform variables: {len(df):,}")
print(f"Passing all thresholds before cap: {len(passing):,}")
print(f"Final seed candidates after cap: {len(selected):,}")
print(
    f"Removed only because of max_candidates cap: "
    f"{max(len(passing) - len(selected), 0):,}"
)

threshold_summary = pd.DataFrame([
    {
        "criterion": column,
        "threshold": threshold,
        "n_passing_individually": int(df[f"pass_{column}"].sum()),
        "n_failing_individually": int((~df[f"pass_{column}"]).sum()),
    }
    for column, (_, threshold) in checks.items()
    if threshold is not None
])

display(threshold_summary)

print("\nSelected candidates by feature group:")
display(
    selected["feature_group"]
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="n_candidates")
)

print("\nSelected candidates by prep root:")
display(
    selected["feature_source"]
    .value_counts()
    .rename_axis("feature_source")
    .reset_index(name="n_candidates")
)

display(
    selected[
        [
            "seed_rank",
            "feature_source",
            "feature_group",
            "feature",
            "selected_transform_mode",
            "oof_metric",
            "delta_clinical",
            "fold_sd",
            "direction_consistency",
            "nonmissing_fraction",
            "candidate_evidence_score",
        ]
    ].head(50)
)

Total best-transform variables: 21,529
Passing all thresholds before cap: 304
Final seed candidates after cap: 50
Removed only because of max_candidates cap: 254


,criterion,threshold,n_passing_individually,n_failing_individually
0,oof_metric,0.75,375,21154
1,fold_sd,0.02,7688,13841
2,direction_consistency,0.80,2066,19463
3,nonmissing_fraction,0.50,21529,0
4,valid_folds,4.00,21529,0



Selected candidates by feature group:


,feature_group,n_candidates
0,triads,33
1,athena,17



Selected candidates by prep root:


,feature_source,n_candidates
0,AR_state,23
1,compartment_state,10
2,compartment,9
3,phenotype_only,8


,seed_rank,feature_source,feature_group,feature,selected_transform_mode,oof_metric,delta_clinical,fold_sd,direction_consistency,nonmissing_fraction,candidate_evidence_score
0,1,AR_state,triads,triad_centered__macrophage_checkpoint_neg__cd8...,log1p_zscore,0.822024,0.305952,0.003260,1.00,1.0,0.972099
1,2,compartment,athena,shannon_phenotype_use_radius__Tumor__median,zscore,0.825595,0.307738,0.004514,1.00,1.0,0.971697
2,3,compartment,athena,renyi_phenotype_use_q3.0_radius__Tumor__mean,zscore,0.816667,0.303571,0.003393,1.00,1.0,0.971684
3,4,compartment,athena,renyi_phenotype_use_q2.0_radius__Tumor__mean,zscore,0.817262,0.311905,0.003993,1.00,1.0,0.971661
4,5,phenotype_only,athena,renyi_phenotype_use_q2.0_radius__Tumor__mean,zscore,0.819048,0.282738,0.002490,1.00,1.0,0.971628
5,6,AR_state,triads,triad_centered__stroma_checkpoint_neg__stroma_...,log1p_zscore,0.823810,0.317857,0.002490,0.92,1.0,0.971436
6,7,compartment_state,triads,triad_centered__stroma_checkpoint_neg__stroma_...,log1p_zscore,0.823810,0.317857,0.002490,0.92,1.0,0.971436
7,8,compartment,athena,renyi_phenotype_use_q2.0_radius__Tumor__median,zscore,0.822024,0.294643,0.003880,0.96,1.0,0.971331
8,9,AR_state,triads,triad_centered__stroma_checkpoint_neg__macroph...,log1p_zscore,0.822619,0.317857,0.004514,0.96,1.0,0.971327
9,10,phenotype_only,athena,renyi_phenotype_use_q3.0_radius__Tumor__mean,zscore,0.817262,0.283929,0.003993,1.00,1.0,0.971097


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2a5_microcompression_sensitivity"
)

profiles = ["conservative", "balanced", "permissive"]

summaries = []

for profile in profiles:
    x = pd.read_csv(
        ROOT / profile / "all_context_compression_summary.csv"
    )
    x["profile"] = profile
    summaries.append(x)

summary = pd.concat(summaries, ignore_index=True)

display(
    summary.pivot_table(
        index=["context_id", "panel"],
        columns="profile",
        values="n_final_representatives",
        aggfunc="first",
    )
)

print("Overall compression:")
display(
    summary.groupby(["profile", "panel"], as_index=False)
    .agg(
        n_seed_features=("n_seed_features", "sum"),
        n_final_representatives=("n_final_representatives", "sum"),
        median_compression_fraction=(
            "compression_fraction_seed_to_final",
            "median",
        ),
        state_removed=("n_state_removed", "sum"),
        metric_removed=("n_metric_removed", "sum"),
        compartment_removed=("n_compartment_removed", "sum"),
        residual_removed=("n_residual_removed", "sum"),
    )
)

,profile,balanced,conservative,permissive
context_id,panel,,,
BLASST__AR__any_response__TURBT__all__median,AR,66,70,59
BLASST__AR__complete_response__TURBT__all__median,AR,42,44,41
BLASST__BT__any_response__TURBT__all__median,BT,39,40,39
BLASST__BT__complete_response__TURBT__all__median,BT,81,83,76
NAC2020__AR__OS__TURBT__all__median,AR,85,90,78
NAC2020__AR__RFS__TURBT__all__median,AR,85,88,79
NAC2020__AR__any_response__TURBT__all__median,AR,17,17,17
NAC2020__AR__complete_response__TURBT__all__median,AR,89,91,83
NAC2020__BT__OS__TURBT__all__median,BT,65,69,61


Overall compression:


,profile,panel,n_seed_features,n_final_representatives,median_compression_fraction,state_removed,metric_removed,compartment_removed,residual_removed
0,balanced,AR,1017,770,0.790000,57,89,28,14
1,balanced,BT,995,775,0.780522,0,114,36,18
2,conservative,AR,1017,805,0.830000,33,67,11,9
3,conservative,BT,995,803,0.815714,0,77,27,17
4,permissive,AR,1017,711,0.740000,92,119,77,16
5,permissive,BT,995,745,0.748626,0,145,50,18


In [ ]:
from itertools import combinations

manifests = {}

for profile in profiles:
    manifests[profile] = pd.read_csv(
        ROOT
        / profile
        / "global_module_candidate_manifest_after_microcompression.csv"
    )

rows = []

for panel in ["AR", "BT"]:
    for a, b in combinations(profiles, 2):
        set_a = set(
            zip(
                manifests[a].loc[
                    manifests[a]["panel"] == panel,
                    "context_id",
                ],
                manifests[a].loc[
                    manifests[a]["panel"] == panel,
                    "feature_uid",
                ],
            )
        )

        set_b = set(
            zip(
                manifests[b].loc[
                    manifests[b]["panel"] == panel,
                    "context_id",
                ],
                manifests[b].loc[
                    manifests[b]["panel"] == panel,
                    "feature_uid",
                ],
            )
        )

        union = set_a | set_b
        jaccard = len(set_a & set_b) / len(union) if union else 1.0

        rows.append(
            {
                "panel": panel,
                "profile_a": a,
                "profile_b": b,
                "n_a": len(set_a),
                "n_b": len(set_b),
                "n_shared": len(set_a & set_b),
                "jaccard": jaccard,
            }
        )

display(pd.DataFrame(rows))

In [1]:
from pathlib import Path
import pandas as pd

SENS_ROOT = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2b_cap_sensitivity"
)

ROOTS = {
    10: SENS_ROOT / "prepared" / "cap010__rho0p9",
    15: SENS_ROOT / "prepared" / "cap015__rho0p9",
    20: SENS_ROOT / "prepared" / "cap020__rho0p9",
}

ontology_rows = []

for cap, root in ROOTS.items():

    for panel in ["AR", "BT"]:

        p = root / panel / f"{panel}_feature_ontology.csv"

        ont = pd.read_csv(p)
        ont["cap"] = cap
        ont["panel"] = panel

        ontology_rows.append(ont)

ontology = pd.concat(
    ontology_rows,
    ignore_index=True
)

audit = (
    ontology
    .groupby(["panel", "cap"], as_index=False)
    .agg(
        n_features=("feature_uid", "nunique"),
        n_zero_cells=("n_cells_parsed", lambda x: (x == 0).sum()),
        frac_zero_cells=("n_cells_parsed", lambda x: (x == 0).mean()),
        n_zero_tissues=("n_tissues_parsed", lambda x: (x == 0).sum()),
        frac_zero_tissues=("n_tissues_parsed", lambda x: (x == 0).mean()),
        n_zero_metrics=("n_metric_families_parsed", lambda x: (x == 0).sum()),
        frac_zero_metrics=("n_metric_families_parsed", lambda x: (x == 0).mean()),
    )
)

display(audit)

,panel,cap,n_features,n_zero_cells,frac_zero_cells,n_zero_tissues,frac_zero_tissues,n_zero_metrics,frac_zero_metrics
0,AR,10,87,1,0.011494,56,0.643678,0,0.0
1,AR,15,127,3,0.023622,76,0.598425,0,0.0
2,AR,20,159,3,0.018868,92,0.578616,0,0.0
3,BT,10,89,0,0.000000,48,0.539326,0,0.0
4,BT,15,125,0,0.000000,65,0.520000,0,0.0
5,BT,20,160,1,0.006250,85,0.531250,0,0.0


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

GRID_ROOT = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2a5_cap_rho_grid_v1"
)

seed_map = pd.read_parquet(
    GRID_ROOT / "all_grid_seed_to_final_representative.parquet"
)

final_manifest = pd.read_parquet(
    GRID_ROOT / "all_grid_context_manifest_deduplicated.parquet"
)

def split_uid(uid):
    parts = str(uid).split("|", 2)
    if len(parts) == 3:
        return parts
    return ["", "", str(uid)]


rows = []

for cap in [10, 15, 20]:
    for panel in ["AR", "BT"]:

        x = seed_map[
            seed_map["candidate_cap"].eq(cap)
            & seed_map["semantic_rho"].eq(0.90)
            & seed_map["panel"].eq(panel)
        ].copy()

        x[
            ["seed_source", "seed_group", "seed_feature"]
        ] = x["seed_feature_uid"].apply(
            lambda z: pd.Series(split_uid(z))
        )

        x[
            ["final_source", "final_group", "final_feature"]
        ] = x["final_representative_uid"].apply(
            lambda z: pd.Series(split_uid(z))
        )

        x["changed"] = (
            x["seed_feature_uid"] != x["final_representative_uid"]
        )

        rows.append({
            "panel": panel,
            "cap": cap,
            "n_seed_records": len(x),
            "n_unique_seed_features":
                x["seed_feature_uid"].nunique(),
            "n_unique_final_representatives":
                x["final_representative_uid"].nunique(),
            "n_changed":
                x["changed"].sum(),
            "fraction_changed":
                x["changed"].mean(),
        })

compression_overview = pd.DataFrame(rows)

display(compression_overview)

,panel,cap,n_seed_records,n_unique_seed_features,n_unique_final_representatives,n_changed,fraction_changed
0,AR,10,110,99,87,16,0.145455
1,BT,10,120,118,89,33,0.275000
2,AR,15,165,145,127,28,0.169697
3,BT,15,180,168,125,49,0.272222
4,AR,20,217,190,160,40,0.184332
5,BT,20,240,216,161,65,0.270833


In [5]:
from pathlib import Path
import pandas as pd

SENS_ROOT = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2b_cap_sensitivity"
)

ROOTS = {
    10: SENS_ROOT / "prepared" / "cap010__rho0p9",
    15: SENS_ROOT / "prepared" / "cap015__rho0p9",
    20: SENS_ROOT / "prepared" / "cap020__rho0p9",
}

ontology_rows = []

for cap, root in ROOTS.items():

    for panel in ["AR", "BT"]:

        p = root / panel / f"{panel}_feature_ontology.csv"

        ont = pd.read_csv(p)
        ont["cap"] = cap
        ont["panel"] = panel

        ontology_rows.append(ont)

ontology = pd.concat(
    ontology_rows,
    ignore_index=True
)

audit = (
    ontology
    .groupby(["panel", "cap"], as_index=False)
    .agg(
        n_features=("feature_uid", "nunique"),
        n_zero_cells=("n_cells_parsed", lambda x: (x == 0).sum()),
        frac_zero_cells=("n_cells_parsed", lambda x: (x == 0).mean()),
        n_zero_tissues=("n_tissues_parsed", lambda x: (x == 0).sum()),
        frac_zero_tissues=("n_tissues_parsed", lambda x: (x == 0).mean()),
        n_zero_metrics=("n_metric_families_parsed", lambda x: (x == 0).sum()),
        frac_zero_metrics=("n_metric_families_parsed", lambda x: (x == 0).mean()),
    )
)

display(audit)

,panel,cap,n_features,n_zero_cells,frac_zero_cells,n_zero_tissues,frac_zero_tissues,n_zero_metrics,frac_zero_metrics
0,AR,10,87,1,0.011494,56,0.643678,0,0.0
1,AR,15,127,3,0.023622,76,0.598425,0,0.0
2,AR,20,159,3,0.018868,92,0.578616,0,0.0
3,BT,10,89,0,0.000000,48,0.539326,0,0.0
4,BT,15,125,0,0.000000,65,0.520000,0,0.0
5,BT,20,160,1,0.006250,85,0.531250,0,0.0


In [3]:
for panel in ["AR", "BT"]:

    x = seed_map[
        seed_map["candidate_cap"].eq(20)
        & seed_map["semantic_rho"].eq(0.90)
        & seed_map["panel"].eq(panel)
    ].copy()

    x = x[
        x["seed_feature_uid"] != x["final_representative_uid"]
    ]

    x[
        ["seed_source", "seed_group", "seed_feature"]
    ] = x["seed_feature_uid"].apply(
        lambda z: pd.Series(split_uid(z))
    )

    x[
        ["final_source", "final_group", "final_feature"]
    ] = x["final_representative_uid"].apply(
        lambda z: pd.Series(split_uid(z))
    )


    print("\n", "=" * 90)
    print(panel, "CAP 20 — CHANGED REPRESENTATIVES")
    print("=" * 90)

    # display(
    #     x[
    #         [
    #             "cohort",
    #             "endpoint",
    #             "context_id",
    #             "seed_source",
    #             "seed_group",
    #             "seed_feature",
    #             "final_source",
    #             "final_group",
    #             "final_feature",
    #         ]
    #     ].sort_values(
    #         ["cohort", "endpoint", "seed_feature"]
    #     )
    # )


AR CAP 20 — CHANGED REPRESENTATIVES

BT CAP 20 — CHANGED REPRESENTATIVES


In [6]:
from pathlib import Path
import pandas as pd

OUTDIR = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2b_cap_sensitivity/"
    "parser_review"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# Useful ordering
sort_cols = [
    c for c in [
        "cap",
        "feature_source",
        "feature_group",
        "feature",
    ]
    if c in ontology.columns
]

for panel in ["AR", "BT"]:

    x = (
        ontology.loc[ontology["panel"].eq(panel)]
        .drop_duplicates()
        .sort_values(sort_cols)
        .copy()
    )

    # Helpful review flags
    x["parser_missing_cell"] = x["n_cells_parsed"].eq(0)
    x["parser_missing_tissue"] = x["n_tissues_parsed"].eq(0)
    x["parser_missing_metric"] = x["n_metric_families_parsed"].eq(0)

    outpath = OUTDIR / f"{panel}_feature_ontology_caps10_15_20.csv"
    x.to_csv(outpath, index=False)

    print(
        f"{panel}: saved {len(x):,} rows -> {outpath}"
    )

    print(
        x[
            [
                "parser_missing_cell",
                "parser_missing_tissue",
                "parser_missing_metric",
            ]
        ].mean()
    )
    print()

AR: saved 373 rows -> /projects/ovcare/users/nikolay_alabi/immuno/stage2_global_modules_v8/stage2b_cap_sensitivity/parser_review/AR_feature_ontology_caps10_15_20.csv
parser_missing_cell      0.018767
parser_missing_tissue    0.600536
parser_missing_metric    0.000000
dtype: float64

BT: saved 374 rows -> /projects/ovcare/users/nikolay_alabi/immuno/stage2_global_modules_v8/stage2b_cap_sensitivity/parser_review/BT_feature_ontology_caps10_15_20.csv
parser_missing_cell      0.002674
parser_missing_tissue    0.529412
parser_missing_metric    0.000000
dtype: float64



In [7]:
# ============================================================
# Audit the CURRENT Stage 2B feature-name parser
# ============================================================

from pathlib import Path
import importlib.util
import re
import pandas as pd
import numpy as np

MODULE_DIR = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/manuscript/modules"
)

GM_PATH = MODULE_DIR / "stage2_global_module_utils_v7.py"

spec = importlib.util.spec_from_file_location(
    "gm_parser_audit",
    GM_PATH,
)
gm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gm)

print("Loaded parser from:")
print(GM_PATH)


# ------------------------------------------------------------
# Unique features to audit
# ------------------------------------------------------------

cols_to_keep = [
    c for c in [
        "panel",
        "cap",
        "feature_uid",
        "feature_source",
        "feature_group",
        "feature",
    ]
    if c in ontology.columns
]

features = (
    ontology[cols_to_keep]
    .drop_duplicates()
    .copy()
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

CHECKPOINT_PATTERN = re.compile(
    r"(?i)(PD[-_ ]?1|PD[-_ ]?L1|"
    r"checkpoint|"
    r"CTLA[-_ ]?4|"
    r"LAG[-_ ]?3|"
    r"TIM[-_ ]?3|"
    r"TIGIT)"
)

STATE_PATTERN = re.compile(
    r"(?i)(state|checkpoint|PD[-_ ]?1|PD[-_ ]?L1)"
)

def join_set(x):
    return ";".join(sorted(map(str, x))) if x else ""

def parse_one(row):

    uid = str(row["feature_uid"])
    raw_feature = str(row.get("feature", ""))

    # Exact functions currently used by Stage 2B ontology
    cells_uid = gm.extract_cells_from_feature(uid)
    tissues_uid = gm.extract_tissues_from_feature(uid)
    metrics_uid = gm.extract_metric_families_from_feature(
        uid,
        feature_group=row.get("feature_group", None),
    )

    # Parse raw feature separately as a sanity check
    cells_raw = gm.extract_cells_from_feature(raw_feature)
    tissues_raw = gm.extract_tissues_from_feature(raw_feature)
    metrics_raw = gm.extract_metric_families_from_feature(
        raw_feature,
        feature_group=row.get("feature_group", None),
    )

    cells_uid_s = join_set(cells_uid)
    cells_raw_s = join_set(cells_raw)

    return pd.Series({

        # Current Stage 2B parser output
        "parsed_cells": cells_uid_s,
        "parsed_tissues": join_set(tissues_uid),
        "parsed_metrics": join_set(metrics_uid),

        "n_parsed_cells": len(cells_uid),
        "n_parsed_tissues": len(tissues_uid),
        "n_parsed_metrics": len(metrics_uid),

        # Parsing the feature name without prep-root/group prefix
        "raw_feature_cells": cells_raw_s,
        "raw_feature_tissues": join_set(tissues_raw),
        "raw_feature_metrics": join_set(metrics_raw),

        # Useful disagreement check
        "uid_vs_raw_cell_disagreement":
            cells_uid != cells_raw,

        # Does feature contain checkpoint/state language?
        "contains_checkpoint_term":
            bool(CHECKPOINT_PATTERN.search(raw_feature)),

        "contains_state_term":
            bool(STATE_PATTERN.search(raw_feature)),

        # Potentially problematic:
        # state/checkpoint tokens appear inside the CELL ontology
        "state_like_token_parsed_as_cell":
            any(
                (
                    "state" in str(c).lower()
                    or "pd1" in str(c).lower()
                    or "pdl1" in str(c).lower()
                    or "checkpoint" in str(c).lower()
                )
                for c in cells_uid
            ),

        "no_cell_parsed":
            len(cells_uid) == 0,

        "no_tissue_parsed":
            len(tissues_uid) == 0,

        "no_metric_parsed":
            len(metrics_uid) == 0,
    })


parser_results = features.apply(parse_one, axis=1)

parser_audit = pd.concat(
    [
        features.reset_index(drop=True),
        parser_results.reset_index(drop=True),
    ],
    axis=1,
)


# ------------------------------------------------------------
# Overall audit summary
# ------------------------------------------------------------

summary = (
    parser_audit
    .groupby(["panel", "cap"], as_index=False)
    .agg(
        n_features=("feature_uid", "nunique"),

        n_no_cell=("no_cell_parsed", "sum"),
        frac_no_cell=("no_cell_parsed", "mean"),

        n_no_tissue=("no_tissue_parsed", "sum"),
        frac_no_tissue=("no_tissue_parsed", "mean"),

        n_no_metric=("no_metric_parsed", "sum"),
        frac_no_metric=("no_metric_parsed", "mean"),

        n_checkpoint_features=("contains_checkpoint_term", "sum"),

        n_state_as_cell=("state_like_token_parsed_as_cell", "sum"),
        frac_state_as_cell=("state_like_token_parsed_as_cell", "mean"),

        n_uid_raw_disagreement=(
            "uid_vs_raw_cell_disagreement",
            "sum",
        ),
    )
)

display(summary)


# ------------------------------------------------------------
# Show suspicious examples
# ------------------------------------------------------------

suspicious = parser_audit[
    parser_audit[
        [
            "no_cell_parsed",
            "state_like_token_parsed_as_cell",
            "uid_vs_raw_cell_disagreement",
        ]
    ].any(axis=1)
].copy()

print(
    f"Suspicious parser rows: "
    f"{len(suspicious):,} / {len(parser_audit):,}"
)

display(
    suspicious[
        [
            c for c in [
                "panel",
                "cap",
                "feature_source",
                "feature_group",
                "feature",
                "parsed_cells",
                "parsed_tissues",
                "parsed_metrics",
                "contains_checkpoint_term",
                "state_like_token_parsed_as_cell",
                "uid_vs_raw_cell_disagreement",
            ]
            if c in suspicious.columns
        ]
    ].head(100)
)


# ------------------------------------------------------------
# Save panel-specific CSVs
# ------------------------------------------------------------

for panel in ["AR", "BT"]:

    x = (
        parser_audit[
            parser_audit["panel"].eq(panel)
        ]
        .sort_values(
            [
                c for c in [
                    "cap",
                    "feature_source",
                    "feature_group",
                    "feature",
                ]
                if c in parser_audit.columns
            ]
        )
    )

    outpath = (
        OUTDIR /
        f"{panel}_feature_parser_audit_caps10_15_20.csv"
    )

    x.to_csv(outpath, index=False)

    print(f"SAVED: {outpath}")

summary.to_csv(
    OUTDIR / "feature_parser_audit_summary.csv",
    index=False,
)

print(
    "SAVED:",
    OUTDIR / "feature_parser_audit_summary.csv",
)

Loaded parser from:
/projects/ovcare/users/nikolay_alabi/immuno/manuscript/modules/stage2_global_module_utils_v7.py


,panel,cap,n_features,n_no_cell,frac_no_cell,n_no_tissue,frac_no_tissue,n_no_metric,frac_no_metric,n_checkpoint_features,n_state_as_cell,frac_state_as_cell,n_uid_raw_disagreement
0,AR,10,87,1,0.011494,56,0.643678,0,0.0,52,33,0.379310,0
1,AR,15,127,3,0.023622,76,0.598425,0,0.0,78,51,0.401575,0
2,AR,20,159,3,0.018868,92,0.578616,0,0.0,100,67,0.421384,0
3,BT,10,89,0,0.000000,48,0.539326,0,0.0,0,0,0.000000,0
4,BT,15,125,0,0.000000,65,0.520000,0,0.0,0,0,0.000000,0
5,BT,20,160,1,0.006250,85,0.531250,0,0.0,0,0,0.000000,0


Suspicious parser rows: 159 / 747


,panel,cap,feature_source,feature_group,feature,parsed_cells,parsed_tissues,parsed_metrics,contains_checkpoint_term,state_like_token_parsed_as_cell,uid_vs_raw_cell_disagreement
0,AR,10,AR_checkpoint_state,athena,renyi_phenotype_use_q0.5_radius__All__max,,,ATHENA_diversity,False,False,False
1,AR,10,AR_checkpoint_state,cell_features,All__count__checkpoint_neg,checkpoint_neg_state,,cell_features,True,True,False
2,AR,10,AR_checkpoint_state,triads,triad__All__center__checkpoint_neg__n_cells,checkpoint_neg_state,,triads,True,True,False
3,AR,10,AR_state,NN,cd8_t_cell__PD1_PDL1_to_macrophage__PD1_Mean,PD1_PDL1_state;PD1_state;PDL1_state;cd8_t_cell...,,NN,True,True,False
4,AR,10,AR_state,NN,cd8_t_cell__PD1_to_macrophage__PD1_SD,PD1_state;cd8_t_cell;macrophage;t_cell,,NN,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...
435,AR,20,AR_state,NN,cd8_t_cell__PD1_to_macrophage__PD1_SD,PD1_state;cd8_t_cell;macrophage;t_cell,,NN,True,True,False
436,AR,20,AR_state,NN,macrophage__PDL1_to_macrophage__checkpoint_neg...,PDL1_state;checkpoint_neg_state;macrophage,,NN,True,True,False
437,AR,20,AR_state,NN,macrophage__PDL1_to_tumor__checkpoint_neg_Max,PDL1_state;checkpoint_neg_state;macrophage;tum...,Tumor,NN,True,True,False
438,AR,20,AR_state,NN,macrophage__checkpoint_neg_to_macrophage__chec...,checkpoint_neg_state;macrophage,,NN,True,True,False


SAVED: /projects/ovcare/users/nikolay_alabi/immuno/stage2_global_modules_v8/stage2b_cap_sensitivity/parser_review/AR_feature_parser_audit_caps10_15_20.csv
SAVED: /projects/ovcare/users/nikolay_alabi/immuno/stage2_global_modules_v8/stage2b_cap_sensitivity/parser_review/BT_feature_parser_audit_caps10_15_20.csv
SAVED: /projects/ovcare/users/nikolay_alabi/immuno/stage2_global_modules_v8/stage2b_cap_sensitivity/parser_review/feature_parser_audit_summary.csv


In [11]:
# ============================================================
# Stage 2A-4 v2.2 DRY-RUN QC
# ============================================================

from pathlib import Path
import pandas as pd

DRY_ROOT = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v9/stage2a4_rescue_parser_dryrun_v2_2"
)

CONTEXT_INDEX = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v8/stage2a_steps1_3_context_review_v4/"
    "stage2a_context_index.csv"
)

index = pd.read_csv(CONTEXT_INDEX)

print("=" * 90)
print("1. CONTEXT COMPLETION")
print("=" * 90)

expected = len(index)
context_dirs = list((DRY_ROOT / "contexts").glob("*"))

done_files = list((DRY_ROOT / "contexts").glob("*/.done"))
summary_files = list(
    (DRY_ROOT / "contexts").glob("*/context_stage2a4_summary.csv")
)

print(f"Expected contexts : {expected}")
print(f"Context dirs      : {len(context_dirs)}")
print(f".done files       : {len(done_files)}")
print(f"Summary files     : {len(summary_files)}")

if len(done_files) == expected:
    print("✓ All contexts have .done files")
else:
    print("⚠ Some contexts are not complete")


# ============================================================
# Combine context summaries
# ============================================================

summary_parts = []

for p in summary_files:
    x = pd.read_csv(p)
    x["context_slug"] = p.parent.name
    summary_parts.append(x)

summary = (
    pd.concat(summary_parts, ignore_index=True)
    if summary_parts
    else pd.DataFrame()
)

if not summary.empty:
    print("\nStatus counts:")
    display(
        summary["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="n_contexts")
    )


# ============================================================
# 2. REMAINING PARSER PROBLEMS
# ============================================================

print("\n" + "=" * 90)
print("2. PARSER PROBLEMS")
print("=" * 90)

problem_files = list(
    (DRY_ROOT / "contexts").glob(
        "*/feature_parser_problem_features.csv"
    )
)

problem_parts = []

for p in problem_files:
    x = pd.read_csv(p)
    if len(x):
        x["context_slug"] = p.parent.name
        problem_parts.append(x)

if problem_parts:
    problems = pd.concat(problem_parts, ignore_index=True)

    print(
        f"⚠ Parser problems remain: "
        f"{len(problems)} rows across "
        f"{problems['context_slug'].nunique()} contexts"
    )

    unique_problems = (
        problems[
            [
                "panel",
                "feature_source",
                "feature_group",
                "feature",
                "parser_status",
                "parser_warnings",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "panel",
                "feature_source",
                "feature_group",
                "feature",
            ]
        )
    )

    print(
        f"Unique problematic feature definitions: "
        f"{len(unique_problems)}"
    )

    display(unique_problems)

else:
    problems = pd.DataFrame()
    print("✓ ZERO remaining parser problems")


# ============================================================
# 3. HARD-STOP CHECK: selected seeds failing parser
# ============================================================

print("\n" + "=" * 90)
print("3. SELECTED-SEED PARSER FAILURES")
print("=" * 90)

seed_error_files = list(
    (DRY_ROOT / "contexts").glob(
        "*/ERROR_selected_seed_parser_failures.csv"
    )
)

if seed_error_files:
    print(
        f"❌ HARD STOP: {len(seed_error_files)} contexts "
        f"have selected-seed parser failures"
    )

    seed_errors = pd.concat(
        [
            pd.read_csv(p).assign(context_slug=p.parent.name)
            for p in seed_error_files
        ],
        ignore_index=True,
    )

    display(
        seed_errors[
            [
                c for c in [
                    "context_slug",
                    "panel",
                    "feature_source",
                    "feature_group",
                    "feature",
                    "parser_status",
                    "parser_warnings",
                ]
                if c in seed_errors.columns
            ]
        ]
    )

else:
    print("✓ ZERO selected-seed parser failures")


# ============================================================
# 4. TECHNICAL VARIABLES
# ============================================================

print("\n" + "=" * 90)
print("4. TECHNICAL / NON-CANDIDATE FEATURES")
print("=" * 90)

technical_files = list(
    (DRY_ROOT / "contexts").glob(
        "*/technical_non_candidate_features.csv"
    )
)

technical_parts = []

for p in technical_files:
    x = pd.read_csv(p)

    if len(x):
        x["context_slug"] = p.parent.name
        technical_parts.append(x)

if technical_parts:

    technical = pd.concat(
        technical_parts,
        ignore_index=True,
    )

    technical_unique = (
        technical[
            [
                c for c in [
                    "panel",
                    "feature_source",
                    "feature_group",
                    "feature",
                    "parsed_feature_type",
                    "parsed_feature_subtype",
                    "candidate_eligible",
                    "parser_status",
                ]
                if c in technical.columns
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                c for c in [
                    "panel",
                    "feature_group",
                    "feature",
                ]
                if c in technical.columns
            ]
        )
    )

    print(
        f"Technical rows across contexts: {len(technical):,}"
    )
    print(
        f"Unique technical definitions: "
        f"{len(technical_unique)}"
    )

    display(technical_unique)

else:
    technical = pd.DataFrame()
    print("No technical_non_candidate_features.csv rows found.")


# ============================================================
# FINAL VERDICT
# ============================================================

print("\n" + "=" * 90)
print("FINAL DRY-RUN VERDICT")
print("=" * 90)

all_done = len(done_files) == expected
no_parser_problems = len(problem_parts) == 0
no_seed_errors = len(seed_error_files) == 0

if all_done and no_parser_problems and no_seed_errors:
    print("✅ PASS")
    print(
        "Parser/rescue dry run looks clean. "
        "Ready for the full Stage 2A-4 v2.2 run."
    )
else:
    print("⚠ NOT READY YET")

    if not all_done:
        print("- Some contexts have not completed.")

    if not no_parser_problems:
        print("- Remaining feature grammars need review.")

    if not no_seed_errors:
        print("- At least one selected seed failed parsing.")

1. CONTEXT COMPLETION
Expected contexts : 28
Context dirs      : 28
.done files       : 28
Summary files     : 28
✓ All contexts have .done files

Status counts:


,status,n_contexts
0,dry_run_registry_only,24
1,excluded_by_manual_rule,4



2. PARSER PROBLEMS
✓ ZERO remaining parser problems

3. SELECTED-SEED PARSER FAILURES
✓ ZERO selected-seed parser failures

4. TECHNICAL / NON-CANDIDATE FEATURES
Technical rows across contexts: 582
Unique technical definitions: 49


,panel,feature_source,feature_group,feature,parsed_feature_type,parsed_feature_subtype,candidate_eligible,parser_status
20,AR,AR_checkpoint_state,cell_features,All__n_cells,technical_qc,n_cells,False,ok
23,AR,AR_state,cell_features,All__n_cells,technical_qc,n_cells,False,ok
26,AR,compartment,cell_features,All__n_cells,technical_qc,n_cells,False,ok
29,AR,compartment_state,cell_features,All__n_cells,technical_qc,n_cells,False,ok
32,AR,phenotype_only,cell_features,All__n_cells,technical_qc,n_cells,False,ok
21,AR,AR_checkpoint_state,cell_features,All__n_resolved_for_ratio,technical_qc,n_resolved_for_ratio,False,ok
24,AR,AR_state,cell_features,All__n_resolved_for_ratio,technical_qc,n_resolved_for_ratio,False,ok
27,AR,compartment,cell_features,All__n_resolved_for_ratio,technical_qc,n_resolved_for_ratio,False,ok
30,AR,compartment_state,cell_features,All__n_resolved_for_ratio,technical_qc,n_resolved_for_ratio,False,ok
33,AR,phenotype_only,cell_features,All__n_resolved_for_ratio,technical_qc,n_resolved_for_ratio,False,ok



FINAL DRY-RUN VERDICT
✅ PASS
Parser/rescue dry run looks clean. Ready for the full Stage 2A-4 v2.2 run.


In [9]:
import pandas as pd

ROOT = (
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v9/stage2a4_filtered_context_matrices"
)

s = pd.read_csv(
    f"{ROOT}/all_context_stage2a4_summary.csv"
)

cols = [
    "context_id",
    "panel",
    "endpoint",
    "status",
    "n_seed_candidates",
    "n_rescue_candidates",
    "n_candidate_registry",
    "n_matrix_features",
    "n_matrix_features_reused",
    "n_matrix_features_newly_built",
    "n_matrix_build_failed",
]

display(
    s[[c for c in cols if c in s.columns]]
    .sort_values(["panel", "context_id"])
)

print("\nStatus counts:")
display(
    s["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n")
)

print("\nContexts with matrix failures:")
display(
    s[s["n_matrix_build_failed"].fillna(0) > 0]
)

array(['Stroma__n_cells', 'Stroma__n_resolved_for_ratio', 'Epi__n_cells',
       'Epi__n_resolved_for_ratio', 'All__n_cells',
       'All__n_resolved_for_ratio', 'n_cells_total_input'], dtype=object)

In [12]:
import pandas as pd

ROOT = (
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v9/stage2a4_filtered_context_matrices"
)

s = pd.read_csv(
    f"{ROOT}/all_context_stage2a4_summary.csv"
)

cols = [
    "context_id",
    "panel",
    "endpoint",
    "status",
    "n_seed_candidates",
    "n_rescue_candidates",
    "n_candidate_registry",
    "n_matrix_features",
    "n_matrix_features_reused",
    "n_matrix_features_newly_built",
    "n_matrix_build_failed",
]

display(
    s[[c for c in cols if c in s.columns]]
    .sort_values(["panel", "context_id"])
)

print("\nStatus counts:")
display(
    s["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n")
)

print("\nContexts with matrix failures:")
display(
    s[s["n_matrix_build_failed"].fillna(0) > 0]
)

,context_id,panel,endpoint,status,n_seed_candidates,n_rescue_candidates,n_candidate_registry,n_matrix_features,n_matrix_features_reused,n_matrix_features_newly_built,n_matrix_build_failed
0,BLASST__AR__OS__TURBT__all__median,AR,OS,excluded_by_manual_rule,0,0,NaN,0,NaN,NaN,NaN
1,BLASST__AR__RFS__TURBT__all__median,AR,RFS,excluded_by_manual_rule,0,0,NaN,0,NaN,NaN,NaN
2,BLASST__AR__any_response__TURBT__all__median,AR,any_response,complete,100,59,135.0,135,124.0,11.0,0.0
3,BLASST__AR__complete_response__TURBT__all__median,AR,complete_response,complete,100,98,188.0,188,138.0,50.0,0.0
8,NAC2020__AR__OS__TURBT__all__median,AR,OS,complete,100,52,123.0,123,119.0,4.0,0.0
9,NAC2020__AR__RFS__TURBT__all__median,AR,RFS,complete,100,100,170.0,170,134.0,36.0,0.0
10,NAC2020__AR__any_response__TURBT__all__median,AR,any_response,complete,17,3,20.0,20,18.0,2.0,0.0
11,NAC2020__AR__complete_response__TURBT__all__me...,AR,complete_response,complete,100,37,121.0,121,104.0,17.0,0.0
16,No-NAC__AR__OS__TURBT__all__median,AR,OS,complete,100,56,130.0,130,124.0,6.0,0.0
17,No-NAC__AR__RFS__TURBT__all__median,AR,RFS,complete,100,52,123.0,123,122.0,1.0,0.0



Status counts:


,status,n
0,complete,24
1,excluded_by_manual_rule,4



Contexts with matrix failures:


,array_id,context_id,cohort,panel,endpoint,sample_type,patient_subset,agg,context_strength,included,...,n_seed_candidates,n_rescue_candidates,n_matrix_features,n_passing_raw_thresholds,n_candidate_registry,n_matrix_patients,n_matrix_build_success,n_matrix_build_failed,n_matrix_features_reused,n_matrix_features_newly_built


In [13]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/projects/ovcare/users/nikolay_alabi/immuno/"
    "stage2_global_modules_v9/stage2b_cap_sensitivity/"
    "shared_matrix_cache"
)

s = pd.read_csv(ROOT / "shared_cache_context_summary.csv")

display(
    s[
        [
            "cohort",
            "panel",
            "n_requested",
            "n_reused",
            "n_newly_built",
            "n_present_final",
            "n_missing_final",
            "n_patients",
            "matrix_exists",
        ]
    ]
)

print("Matrices:", s["matrix_exists"].sum(), "/", len(s))

f = pd.read_csv(ROOT / "shared_cache_build_failures.csv")

print("\nUnavailable feature × cohort rows:", len(f))

if len(f):
    display(
        f["reason"]
        .value_counts()
        .rename_axis("reason")
        .reset_index(name="n")
    )

,cohort,panel,n_requested,n_reused,n_newly_built,n_present_final,n_missing_final,n_patients,matrix_exists
0,NAC2020,AR,182,155,27,182,0,40,True
1,NAC2020,BT,176,149,27,176,0,38,True
2,PURE01,AR,182,155,27,182,0,70,True
3,PURE01,BT,176,149,27,176,0,66,True
4,BLASST,AR,182,155,27,182,0,37,True
5,BLASST,BT,176,149,27,176,0,39,True
6,No-NAC,AR,182,155,27,182,0,56,True
7,No-NAC,BT,176,149,27,176,0,57,True


Matrices: 8 / 8


EmptyDataError: No columns to parse from file